# 🔥 TextForge AI — LSTM Text Generation Model

**Author:** Tushar Tamrakar  
**Project:** TextForge AI — Production-Grade Generative Text Platform  
**Live App:** https://textforge-ai-sable.vercel.app  
**GitHub:** https://github.com/TUSHARTAMRAKAR/textforge-ai

---

## 📖 Academic Context

This notebook is the **ML foundation** of TextForge AI — a production-grade generative text platform. While the live application uses Google Gemini 2.5 Flash (a Transformer-based LLM), this notebook demonstrates the **fundamental neural network principles** that underpin all modern text generation:

| Component | This Notebook | Production App |
|-----------|--------------|----------------|
| Architecture | Character-level LSTM | Gemini 2.5 Flash (Transformer) |
| Parameters | ~500K | Billions |
| Training data | Curated sentences | Trillions of tokens |
| Output | Basic pattern learning | Human-level coherence |
| Purpose | **Teaching fundamentals** | **Production generation** |

## 🏗 What TextForge AI Builds On Top

The production application extends this ML foundation with:
- **Deep Domain Templates** — 6 domains (Legal, Medical, Startup, Research, Grant, HR) with 70+ parameters
- **AI Detection & Humaniser** — 5-dimension linguistic scoring algorithm
- **Academic Citations** — Real sources from 200M+ papers via Semantic Scholar & CrossRef
- **Bharat AI** — Native generation in Hindi, Marathi, Tamil, Telugu, Bengali, Gujarati
- **Real-time SSE Streaming** — Token-by-token delivery to the browser

---

## 📚 Table of Contents

1. [The Vanishing Gradient Problem](#1-the-vanishing-gradient-problem)
2. [LSTM Architecture & Gates](#2-lstm-architecture--gates)
3. [Environment Setup](#3-environment-setup)
4. [Training Data Preparation](#4-training-data-preparation)
5. [Vocabulary & Tokenisation](#5-vocabulary--tokenisation)
6. [Dataset & DataLoader](#6-dataset--dataloader)
7. [LSTM Model Definition](#7-lstm-model-definition)
8. [Training Loop](#8-training-loop)
9. [Loss Visualisation](#9-loss-visualisation)
10. [Temperature Sampling](#10-temperature-sampling)
11. [Text Generation & Evaluation](#11-text-generation--evaluation)
12. [LSTM vs Transformer — Academic Comparison](#12-lstm-vs-transformer--academic-comparison)
13. [Connection to Production App](#13-connection-to-production-app)

## 1. The Vanishing Gradient Problem

Before LSTMs, vanilla RNNs suffered from the **vanishing gradient problem**.

During Backpropagation Through Time (BPTT), gradients are multiplied at each time step:

$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial h_T} \cdot \prod_{t=1}^{T} \frac{\partial h_t}{\partial h_{t-1}}$$

If $\left|\frac{\partial h_t}{\partial h_{t-1}}\right| < 1$ for all $t$, gradients shrink **exponentially** and the model cannot learn long-range dependencies.

> **Hochreiter & Schmidhuber (1997)** solved this with the **Long Short-Term Memory** network — introducing a cell state that can carry information across hundreds of time steps without degradation.

## 2. LSTM Architecture & Gates

The LSTM introduces three learnable gates that selectively control information flow:

### Forget Gate
$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$
Decides what fraction of the cell state to erase. Output ∈ [0,1]: 0 = forget, 1 = keep.

### Input Gate + Candidate
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
$$\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$$
Decides what new information to write to the cell state.

### Cell State Update
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$
Element-wise: forget old information + write new information.

### Output Gate
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$
$$h_t = o_t \odot \tanh(C_t)$$
Decides what to expose as the hidden state output.

Where $\sigma$ = sigmoid activation, $\odot$ = element-wise product.

## 3. Environment Setup

In [ ]:
# Install dependencies
!pip install torch numpy matplotlib tqdm -q

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random
import os

# Device configuration — uses GPU if available (T4 on Colab)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch version: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## 4. Training Data Preparation

We use a curated corpus covering the domains TextForge AI specialises in: **academic writing, legal language, medical documentation, and startup communication**.

In [ ]:
# Curated training corpus — mirrors TextForge AI domain coverage
# Academic · Legal · Medical · Startup · General

corpus = """
Artificial intelligence represents a fundamental transformation in how machines process and generate human language.
The development of neural networks has enabled computers to learn patterns from vast amounts of text data.
Deep learning models can now generate coherent paragraphs on virtually any topic with remarkable fluency.
Natural language processing has advanced significantly since the introduction of attention mechanisms.
The transformer architecture, introduced by Vaswani et al. in 2017, revolutionised sequence modelling tasks.
Large language models demonstrate emergent capabilities not explicitly programmed by their creators.
Text generation systems require careful prompt engineering to produce consistent, high-quality outputs.
Academic writing demands precision, proper citation, and adherence to disciplinary conventions.
Legal documents require specific jurisdiction clauses, liability limitations, and dispute resolution mechanisms.
Medical case studies follow structured formats including subjective findings, objective data, assessment, and plan.
Startup pitch narratives must communicate the problem, solution, market size, and competitive advantage clearly.
Research abstracts summarise background, methodology, results, and conclusions within strict word limits.
Grant proposals require compelling impact statements and detailed budget justifications.
Human resource documents balance legal compliance with organisational culture and tone.
Machine learning algorithms improve through exposure to labelled training data and gradient-based optimisation.
The vanishing gradient problem prevented early recurrent networks from learning long-range dependencies.
Long short-term memory networks introduced gating mechanisms to selectively retain relevant information.
Character-level language models learn to predict the next character given a sequence of preceding characters.
Word embeddings represent semantic relationships between tokens as dense vectors in continuous space.
Attention mechanisms allow models to focus on relevant parts of the input when generating each output token.
Backpropagation through time computes gradients by unrolling the recurrent network across time steps.
Dropout regularisation prevents overfitting by randomly zeroing activations during training.
Batch normalisation stabilises training by normalising layer inputs to have zero mean and unit variance.
The cross-entropy loss function measures the difference between predicted and actual token distributions.
Gradient clipping prevents exploding gradients by scaling the gradient norm to a maximum threshold.
India is emerging as a global technology powerhouse with a thriving startup ecosystem.
Hindi is spoken by over 500 million people making it one of the most widely used languages in the world.
Tamil Nadu has a rich tradition of literature dating back over two thousand years to the Sangam period.
Maharashtra is home to Mumbai, India's financial capital and the centre of its entertainment industry.
Bengali literature produced Nobel laureate Rabindranath Tagore whose works shaped modern Indian culture.
Gujarat's entrepreneurial spirit has produced global business leaders who transformed entire industries.
The digital India initiative aims to transform the nation into a digitally empowered knowledge economy.
Semantic Scholar indexes over two hundred million academic papers across all scientific disciplines.
CrossRef provides persistent digital object identifiers for scholarly publications worldwide.
APA seventh edition citation format requires the author surname, publication year, and page numbers.
IEEE citation style uses numbered references in square brackets corresponding to a bibliography list.
MLA ninth edition emphasises the works cited page with hanging indentation for each reference entry.
Server-sent events enable real-time unidirectional data streaming from server to browser over HTTP.
MongoDB Atlas provides a fully managed cloud database with automatic scaling and global distribution.
Next.js fourteen introduces the app router with React server components for improved performance.
TypeScript adds static type checking to JavaScript enabling better tooling and fewer runtime errors.
Express.js is a minimal and flexible Node.js web framework ideal for building REST APIs.
OAuth two point zero enables secure delegated authorisation without sharing user credentials.
JSON web tokens provide a compact and self-contained way to securely transmit information between parties.
Rate limiting protects APIs from abuse by restricting the number of requests from each client.
Zod provides TypeScript-first schema validation with automatic type inference for runtime safety.
Vercel deploys Next.js applications to a global edge network with automatic SSL and CDN optimisation.
Railway provides container-based deployment for backend services with automatic GitHub integration.
""".strip()

print(f'Corpus length: {len(corpus):,} characters')
print(f'Unique characters: {len(set(corpus))}')
print(f'Approximate words: {len(corpus.split()):,}')
print(f'\nFirst 200 chars:\n{corpus[:200]}')

## 5. Vocabulary & Tokenisation

We build a **character-level vocabulary** — the model learns to predict the next character given a context window. This approach:
- Requires no pre-built tokeniser
- Naturally handles rare words
- Demonstrates the fundamental principle of all language models

In [ ]:
# Build character-level vocabulary
chars     = sorted(list(set(corpus)))
vocab_size = len(chars)

# Bidirectional mappings: character ↔ integer index
char2idx = {ch: i for i, ch in enumerate(chars)}
idx2char = {i: ch for i, ch in enumerate(chars)}

# Encode entire corpus as integer sequence
encoded = [char2idx[ch] for ch in corpus]

print(f'Vocabulary size: {vocab_size} unique characters')
print(f'Encoded corpus length: {len(encoded):,} tokens')
print(f'\nVocabulary: {" ".join(chars[:30])}...')
print(f'\nSample encoding:')
sample = corpus[:20]
sample_enc = [char2idx[c] for c in sample]
print(f'  Text:    "{sample}"')
print(f'  Encoded: {sample_enc}')

## 6. Dataset & DataLoader

We create overlapping sequences of length `seq_length`. For each sequence, the target is the same sequence shifted one position forward — the model learns to predict the next character.

In [ ]:
class TextDataset(Dataset):
    """
    Character-level text dataset.
    Each sample: (input_sequence, target_sequence)
    where target is input shifted one position forward.
    """
    def __init__(self, encoded_text, seq_length):
        self.data       = torch.tensor(encoded_text, dtype=torch.long)
        self.seq_length = seq_length

    def __len__(self):
        # Each position (except last seq_length) is a valid starting point
        return len(self.data) - self.seq_length

    def __getitem__(self, idx):
        x = self.data[idx     : idx + self.seq_length]       # Input sequence
        y = self.data[idx + 1 : idx + self.seq_length + 1]   # Target (shifted by 1)
        return x, y


# Hyperparameters
SEQ_LENGTH  = 80    # Characters per training sequence
BATCH_SIZE  = 32    # Sequences per gradient update
EMBED_DIM   = 128   # Character embedding dimensions
HIDDEN_DIM  = 256   # LSTM hidden state dimensions
NUM_LAYERS  = 2     # Stacked LSTM layers
DROPOUT     = 0.3   # Dropout probability
EPOCHS      = 30    # Training epochs
LR          = 0.002 # Initial learning rate
CLIP_GRAD   = 5.0   # Gradient clipping threshold

dataset    = TextDataset(encoded, SEQ_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f'Dataset size:    {len(dataset):,} sequences')
print(f'Batches/epoch:   {len(dataloader)}')
print(f'Sequence length: {SEQ_LENGTH} characters')
print(f'Batch size:      {BATCH_SIZE}')
print(f'\nSample batch shapes:')
x_sample, y_sample = next(iter(dataloader))
print(f'  Input  (x): {x_sample.shape}  → [batch, seq_length]')
print(f'  Target (y): {y_sample.shape}  → [batch, seq_length]')

## 7. LSTM Model Definition

Our model architecture:
```
Character index
      ↓
Embedding Layer    (vocab_size → embed_dim=128)
      ↓
LSTM Layer 1       (embed_dim → hidden_dim=256, dropout=0.3)
      ↓
LSTM Layer 2       (hidden_dim → hidden_dim=256)
      ↓
Dropout Layer      (p=0.3)
      ↓
Linear Layer       (hidden_dim=256 → vocab_size)
      ↓
Logits → Temperature Sampling → Next character
```

In [ ]:
class LSTMTextGenerator(nn.Module):
    """
    Character-level LSTM Language Model.

    Architecture:
        Embedding → LSTM (stacked) → Dropout → Linear → Logits

    Args:
        vocab_size  : Number of unique characters
        embed_dim   : Character embedding dimensions
        hidden_dim  : LSTM hidden state size
        num_layers  : Number of stacked LSTM layers
        dropout     : Dropout probability (applied between LSTM layers)
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Layer 1: Embedding — maps integer indices to dense vectors
        # Why: One-hot encoding is sparse (vocab_size dims); embeddings are dense (embed_dim dims)
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # Layer 2: Stacked LSTM with dropout between layers
        # Why stacked: Layer 1 learns local syntax, Layer 2 learns higher-level patterns
        self.lstm = nn.LSTM(
            input_size  = embed_dim,
            hidden_size = hidden_dim,
            num_layers  = num_layers,
            dropout     = dropout,
            batch_first = True   # Input shape: [batch, seq, features]
        )

        # Layer 3: Dropout — Srivastava et al. (2014) regularisation
        self.dropout = nn.Dropout(dropout)

        # Layer 4: Linear projection — maps hidden state to vocabulary logits
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        # x shape: [batch_size, seq_length]
        embedded = self.embedding(x)          # [batch, seq, embed_dim]
        output, hidden = self.lstm(embedded, hidden)  # [batch, seq, hidden_dim]
        output = self.dropout(output)         # [batch, seq, hidden_dim]
        logits = self.fc(output)              # [batch, seq, vocab_size]
        return logits, hidden

    def init_hidden(self, batch_size):
        """Initialise hidden state and cell state to zeros."""
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        return (h0, c0)


# Instantiate model
model = LSTMTextGenerator(
    vocab_size  = vocab_size,
    embed_dim   = EMBED_DIM,
    hidden_dim  = HIDDEN_DIM,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT
).to(device)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTotal trainable parameters: {total_params:,}')

## 8. Training Loop

Key training decisions:
- **Adam optimiser** — adaptive per-parameter learning rates
- **CrossEntropyLoss** — standard for multi-class token prediction
- **Gradient clipping** (threshold=5.0) — **critical for RNNs** to prevent exploding gradients
- **LR scheduler** — halves learning rate every 10 epochs

In [ ]:
criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimiser, step_size=10, gamma=0.5)

train_losses = []

print('Starting training...')
print(f'Device: {device} | Epochs: {EPOCHS} | Batches/epoch: {len(dataloader)}')
print('-' * 60)

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for batch_x, batch_y in dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # Forward pass
        logits, _ = model(batch_x)

        # Reshape for CrossEntropyLoss: [batch*seq, vocab] vs [batch*seq]
        loss = criterion(
            logits.reshape(-1, vocab_size),
            batch_y.reshape(-1)
        )

        # Backward pass
        optimiser.zero_grad()
        loss.backward()

        # Gradient clipping — CRITICAL for RNN stability
        # Without this: gradients can grow exponentially → NaN weights
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)

        optimiser.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    train_losses.append(avg_loss)
    scheduler.step()

    if epoch % 5 == 0 or epoch == 1:
        lr_current = optimiser.param_groups[0]['lr']
        print(f'Epoch [{epoch:2d}/{EPOCHS}] | Loss: {avg_loss:.4f} | LR: {lr_current:.5f}')

print('\nTraining complete!')

## 9. Loss Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('TextForge AI — LSTM Training Analysis', fontsize=14, fontweight='bold')

# Plot 1: Training loss curve
axes[0].plot(range(1, EPOCHS+1), train_losses, color='#D47E30', linewidth=2.5, marker='o', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss Curve')
axes[0].grid(True, alpha=0.3)
axes[0].fill_between(range(1, EPOCHS+1), train_losses, alpha=0.15, color='#D47E30')

# Annotate LR drops
for drop_epoch in [10, 20]:
    if drop_epoch < len(train_losses):
        axes[0].axvline(x=drop_epoch, color='gray', linestyle='--', alpha=0.5)
        axes[0].text(drop_epoch + 0.3, max(train_losses)*0.9, 'LR÷2', fontsize=8, color='gray')

# Plot 2: Loss improvement percentage
if len(train_losses) > 1:
    improvements = [0] + [((train_losses[i-1] - train_losses[i]) / train_losses[i-1]) * 100
                          for i in range(1, len(train_losses))]
    colors = ['#1D9E75' if x >= 0 else '#E05050' for x in improvements]
    axes[1].bar(range(1, EPOCHS+1), improvements, color=colors, alpha=0.8)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss Improvement (%)')
    axes[1].set_title('Per-Epoch Improvement')
    axes[1].axhline(y=0, color='black', linewidth=0.5)
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Initial loss:  {train_losses[0]:.4f}')
print(f'Final loss:    {train_losses[-1]:.4f}')
print(f'Improvement:   {((train_losses[0] - train_losses[-1]) / train_losses[0] * 100):.1f}%')

## 10. Temperature Sampling

Temperature $T$ controls the creativity vs coherence trade-off:

$$P(x_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- **Low T (0.1–0.5):** Sharp distribution → conservative, coherent, predictable
- **High T (1.0–2.0):** Flat distribution → creative, surprising, potentially incoherent
- **TextForge AI production default:** T = 0.7 (balanced)

In [ ]:
def generate_text(model, seed_text, length=200, temperature=0.7):
    """
    Generate text using temperature sampling.

    Args:
        model       : Trained LSTMTextGenerator
        seed_text   : Starting string to condition generation
        length      : Number of characters to generate
        temperature : Sampling temperature (higher = more creative)

    Returns:
        Generated string
    """
    model.eval()
    generated = seed_text
    hidden    = model.init_hidden(batch_size=1)

    # Warm up hidden state on seed text
    with torch.no_grad():
        for ch in seed_text[:-1]:
            if ch not in char2idx:
                continue
            x = torch.tensor([[char2idx[ch]]], dtype=torch.long).to(device)
            _, hidden = model(x, hidden)

        # Generate new characters
        current_char = seed_text[-1]
        for _ in range(length):
            if current_char not in char2idx:
                current_char = ' '

            x      = torch.tensor([[char2idx[current_char]]], dtype=torch.long).to(device)
            logits, hidden = model(x, hidden)

            # Temperature scaling
            scaled_logits = logits[0, 0, :] / temperature
            probs         = torch.softmax(scaled_logits, dim=-1)

            # Multinomial sampling — NOT argmax (which always picks the most likely)
            # Argmax = deterministic, repetitive
            # Multinomial = stochastic, creative
            next_idx     = torch.multinomial(probs, num_samples=1).item()
            current_char = idx2char[next_idx]
            generated   += current_char

    return generated


# Visualise temperature effect
print('Temperature effect demonstration:')
print('=' * 60)
seed = 'Artificial intelligence'

for temp in [0.3, 0.7, 1.2]:
    label = 'Conservative' if temp < 0.5 else 'Balanced' if temp < 1.0 else 'Creative'
    result = generate_text(model, seed, length=80, temperature=temp)
    print(f'\nT={temp} ({label}):')
    print(f'  "{result[:100]}"')

## 11. Text Generation & Evaluation

In [ ]:
# Generate across TextForge AI domains
test_seeds = [
    ("Artificial intelligence",           "General / Academic"),
    ("The legal contract between",         "Legal Domain"),
    ("The patient presented with",         "Medical Domain"),
    ("Our startup has developed",          "Startup Domain"),
    ("India is emerging as",               "Bharat AI Context"),
]

print('TextForge AI — LSTM Generation Samples')
print('=' * 70)
for seed, domain in test_seeds:
    generated = generate_text(model, seed, length=150, temperature=0.7)
    print(f'\n📌 Domain: {domain}')
    print(f'   Seed: "{seed}"')
    print(f'   Output: "{generated[:180]}"')
    print()

In [ ]:
# Perplexity evaluation
# Perplexity = exp(cross-entropy loss)
# Lower is better — indicates the model is less "surprised" by the data

def calculate_perplexity(model, dataloader, criterion):
    model.eval()
    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            logits, _ = model(batch_x)
            loss = criterion(logits.reshape(-1, vocab_size), batch_y.reshape(-1))
            total_loss += loss.item()
            total_batches += 1

    avg_loss   = total_loss / total_batches
    perplexity = np.exp(avg_loss)
    return avg_loss, perplexity

final_loss, perplexity = calculate_perplexity(model, dataloader, criterion)

print('📊 Final Model Evaluation')
print('=' * 40)
print(f'Cross-Entropy Loss : {final_loss:.4f}')
print(f'Perplexity         : {perplexity:.2f}')
print(f'Vocabulary size    : {vocab_size}')
print(f'Parameters         : {total_params:,}')
print()
print('Interpretation:')
print(f'  A perplexity of {perplexity:.0f} means the model is as confused')
print(f'  as if choosing uniformly from ~{perplexity:.0f} equally likely characters.')
print(f'  Random baseline would be: {vocab_size:.0f} (= vocabulary size)')
print(f'  Improvement over random:  {((vocab_size - perplexity) / vocab_size * 100):.1f}%')

## 12. LSTM vs Transformer — Academic Comparison

This notebook implements an LSTM. The production TextForge AI application uses Google Gemini 2.5 Flash, which is based on the Transformer architecture (Vaswani et al., 2017). Understanding why Transformers superseded LSTMs is academically important:

| Dimension | LSTM (This Notebook) | Transformer (Gemini) |
|-----------|---------------------|---------------------|
| **Architecture** | Recurrent (sequential) | Self-attention (parallel) |
| **Complexity** | O(n) sequence length | O(n²) attention — quadratic |
| **Long-range deps** | Cell state gating | Direct attention to any token |
| **Parallelisation** | Sequential — slow training | Fully parallel — fast training |
| **Parameters** | ~500K | Billions |
| **Training data** | 50 curated sentences | Trillions of tokens |
| **Seminal paper** | Hochreiter & Schmidhuber (1997) | Vaswani et al. (2017) |
| **Output quality** | Basic pattern learning | Human-level coherence |

> **Key insight:** The LSTM's sequential bottleneck fundamentally limits its ability to capture very long-range dependencies. The Transformer's attention mechanism addresses this directly — every token can attend to every other token in O(1) operations, regardless of distance. This is why understanding LSTMs makes the Transformer's innovation intuitive.

In [ ]:
# Visualise: LSTM sequential processing vs Transformer parallel attention
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM vs Transformer — Processing Comparison', fontsize=13, fontweight='bold')

# LSTM: Sequential — each step depends on previous
n_tokens = 8
ax = axes[0]
ax.set_title('LSTM: Sequential Processing', fontsize=11)
for i in range(n_tokens):
    ax.add_patch(plt.Rectangle((i*1.2, 1.5), 0.8, 0.8, color='#D47E30', alpha=0.8))
    ax.text(i*1.2+0.4, 1.9, f't{i+1}', ha='center', va='center', color='white', fontsize=9, fontweight='bold')
    if i < n_tokens-1:
        ax.annotate('', xy=((i+1)*1.2, 1.9), xytext=(i*1.2+0.8, 1.9),
                    arrowprops=dict(arrowstyle='->', color='#E8A050', lw=2))
ax.text(n_tokens*1.2/2, 1.0, 'Each token must wait for all previous tokens', ha='center', fontsize=9, color='gray')
ax.set_xlim(-0.2, n_tokens*1.2)
ax.set_ylim(0.5, 3.0)
ax.axis('off')

# Transformer: Parallel — all tokens attend to all others
ax2 = axes[1]
ax2.set_title('Transformer: Parallel Attention', fontsize=11)
positions = [(i*1.2, 1.5) for i in range(n_tokens)]
for i, (x, y) in enumerate(positions):
    ax2.add_patch(plt.Rectangle((x, y), 0.8, 0.8, color='#185FA5', alpha=0.8))
    ax2.text(x+0.4, y+0.4, f't{i+1}', ha='center', va='center', color='white', fontsize=9, fontweight='bold')
# Draw attention connections (sample)
for i in range(0, n_tokens, 2):
    for j in range(0, n_tokens, 3):
        if i != j:
            x1, y1 = positions[i][0]+0.4, positions[i][1]+0.8
            x2, y2 = positions[j][0]+0.4, positions[j][1]+0.8
            ax2.plot([x1, x2], [y1+0.1*(i%2), y2+0.1*(j%2)], 'b-', alpha=0.15, linewidth=0.8)
ax2.text(n_tokens*1.2/2, 1.0, 'All tokens process simultaneously — O(n²) attention', ha='center', fontsize=9, color='gray')
ax2.set_xlim(-0.2, n_tokens*1.2)
ax2.set_ylim(0.5, 3.5)
ax2.axis('off')

plt.tight_layout()
plt.savefig('lstm_vs_transformer.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Connection to Production App

This notebook demonstrates the **theoretical foundation**. Here is how TextForge AI extends these principles into a production system:

In [ ]:
print("""
┌─────────────────────────────────────────────────────────────────────┐
│              TextForge AI — From Notebook to Production             │
├─────────────────────────────┬───────────────────────────────────────┤
│  This Notebook              │  Production App                       │
├─────────────────────────────┼───────────────────────────────────────┤
│  LSTM character model       │  Gemini 2.5 Flash (Transformer)       │
│  Temperature sampling       │  Structured prompt engineering        │
│  Single corpus              │  6 domain templates, 70+ parameters   │
│  Basic generation           │  AI Detection + Humaniser             │
│  No citations               │  Semantic Scholar + CrossRef (200M+)  │
│  English only               │  6 Indian languages (Bharat AI)       │
│  Notebook output            │  Real-time SSE streaming to browser   │
│  No persistence             │  MongoDB Atlas per-user history       │
│  No auth                    │  Google + GitHub OAuth                │
│  No deployment              │  Vercel + Railway (live production)   │
└─────────────────────────────┴───────────────────────────────────────┘

Live Application : https://textforge-ai-sable.vercel.app
Professor Demo   : https://textforge-ai-sable.vercel.app/demo
GitHub           : https://github.com/TUSHARTAMRAKAR/textforge-ai

Built by: Tushar Tamrakar
""")

print('\n📊 Summary Statistics:')
print(f'  Model parameters  : {total_params:,}')
print(f'  Vocabulary size   : {vocab_size} characters')
print(f'  Training sequences: {len(dataset):,}')
print(f'  Final loss        : {train_losses[-1]:.4f}')
print(f'  Training epochs   : {EPOCHS}')
print(f'  Device used       : {device}')
print(f'\n  Production API    : Gemini 2.5 Flash')
print(f'  Production DB     : MongoDB Atlas')
print(f'  Domains supported : Legal, Medical, Startup, Research, Grant, HR')
print(f'  Languages (India) : Hindi, Marathi, Tamil, Telugu, Bengali, Gujarati')
print(f'  Citation sources  : Semantic Scholar (200M+), CrossRef (130M+)')